# Car Price Baseline Pipeline - Real Scale Evaluation

Notebook này giữ nguyên đường dẫn train/val/test, vẫn train bằng `price_log`, nhưng đánh giá thêm trên thang đo giá thực tế bằng inverse log.

In [1]:
# Nếu thiếu thư viện target encoding, chạy cell này trước
# !pip install category_encoders

In [25]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    import category_encoders as ce
    HAS_CE = True
except Exception:
    HAS_CE = False


# =========================
# 1. Load data
# =========================

DATA_DIR = Path("../../data/bonbanh/split")

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)


# =========================
# 2. Define target/features
# =========================

TARGET = "price_log"
REAL_TARGET = "price"

DROP_COLS = [
    "url",
    "date",
    REAL_TARGET,
]

DROP_COLS = [col for col in DROP_COLS if col in train_df.columns]

feature_cols = [
    col for col in train_df.columns
    if col not in DROP_COLS + [TARGET]
]

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].copy()

num_cols = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Drop cols:", DROP_COLS)
print("Target:", TARGET)
print("Features:", len(feature_cols))
print("Numerical:", len(num_cols), num_cols)
print("Categorical:", len(cat_cols), cat_cols)


# =========================
# 3. Inverse log target
# =========================

def infer_log_transform(df, log_col=TARGET, real_col=REAL_TARGET):
    if real_col not in df.columns:
        return "log1p"

    sample = df[[log_col, real_col]].dropna()

    if sample.empty:
        return "log1p"

    y_log = sample[log_col].to_numpy()
    y_real = sample[real_col].to_numpy()

    err_exp = np.nanmedian(np.abs(np.exp(y_log) - y_real))
    err_expm1 = np.nanmedian(np.abs(np.expm1(y_log) - y_real))

    return "log" if err_exp <= err_expm1 else "log1p"


LOG_TRANSFORM = infer_log_transform(train_df)
print("Detected transform:", LOG_TRANSFORM)


def inverse_log_price(y_log):
    y_log = np.asarray(y_log, dtype=float)

    if LOG_TRANSFORM == "log":
        return np.exp(y_log)

    return np.expm1(y_log)


# =========================
# 4. Preprocessors
# =========================

def make_onehot_preprocessor(scale_numeric=True):
    num_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


def make_ordinal_preprocessor(scale_numeric=False):
    num_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


def make_target_preprocessor(scale_numeric=False):
    if not HAS_CE:
        raise ImportError("Bạn cần cài category_encoders: pip install category_encoders")

    num_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("target", ce.TargetEncoder()),
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


# =========================
# 5. Models
# =========================

def get_models():
    models = {
        "linear_regression": LinearRegression(),

        "ridge": Ridge(alpha=1.0, random_state=42),

        "lasso": Lasso(
            alpha=0.001,
            random_state=42,
            max_iter=5000,
        ),

        "elasticnet": ElasticNet(
            alpha=0.001,
            l1_ratio=0.5,
            random_state=42,
            max_iter=5000,
        ),

        "knn_5": KNeighborsRegressor(n_neighbors=5),

        "knn_15": KNeighborsRegressor(n_neighbors=15),

        "decision_tree": DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=5,
            random_state=42,
        ),

        "random_forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),

        "extra_trees": ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),

        "gradient_boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        ),

        "hist_gradient_boosting": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            random_state=42,
        ),
    }

    if HAS_XGB:
        models["xgboost"] = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        )

    if HAS_LGBM:
        models["lightgbm"] = LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )

    return models


# =========================
# 6. Evaluation
# =========================

def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def evaluate_on_dataframe(model_name, encoding, pipeline, split_name, df_split):
    X = df_split[feature_cols].copy()
    y_log_true = df_split[TARGET].copy()

    pred_log = pipeline.predict(X)

    log_metrics = evaluate_regression(y_log_true, pred_log)

    y_real_true = (
        df_split[REAL_TARGET].to_numpy()
        if REAL_TARGET in df_split.columns
        else inverse_log_price(y_log_true)
    )

    pred_real = inverse_log_price(pred_log)

    real_metrics = evaluate_regression(y_real_true, pred_real)

    return {
        "model": model_name,
        "encoding": encoding,
        "split": split_name,
        "n_samples": len(df_split),

        "MAE_log": log_metrics["MAE"],
        "RMSE_log": log_metrics["RMSE"],
        "R2_log": log_metrics["R2"],

        "MAE_real": real_metrics["MAE"],
        "RMSE_real": real_metrics["RMSE"],
        "R2_real": real_metrics["R2"],
    }


def evaluate_pipeline(model_name, encoding, pipeline):
    pipeline.fit(X_train, y_train)

    rows = []

    for split_name, df_split in [
        ("train", train_df),
        ("val", val_df),
        ("test_original", test_df),
    ]:
        rows.append(
            evaluate_on_dataframe(
                model_name=model_name,
                encoding=encoding,
                pipeline=pipeline,
                split_name=split_name,
                df_split=df_split,
            )
        )

    phantom_mask = (
        (test_df["brand"] == "rolls royce") &
        (test_df["model"] == "phantom") &
        (test_df["price"] == 66000000000)
    )

    test_without_phantom_df = test_df[~phantom_mask].copy()

    rows.append(
        evaluate_on_dataframe(
            model_name=model_name,
            encoding=encoding,
            pipeline=pipeline,
            split_name="test_without_phantom_66B",
            df_split=test_without_phantom_df,
        )
    )

    return rows, pipeline


# =========================
# 7. Build experiments
# =========================

models = get_models()
experiments = []

linear_like = [
    "linear_regression",
    "ridge",
    "lasso",
    "elasticnet",
    "knn_5",
    "knn_15",
]

tree_like = [
    "decision_tree",
    "random_forest",
    "extra_trees",
    "gradient_boosting",
    "hist_gradient_boosting",
    "xgboost",
    "lightgbm",
]

for model_name in linear_like:
    if model_name in models:
        experiments.append((
            model_name,
            "onehot_scaled",
            Pipeline([
                ("preprocess", make_onehot_preprocessor(scale_numeric=True)),
                ("model", models[model_name]),
            ]),
        ))

for model_name in tree_like:
    if model_name in models:
        experiments.append((
            model_name,
            "ordinal",
            Pipeline([
                ("preprocess", make_ordinal_preprocessor(scale_numeric=False)),
                ("model", models[model_name]),
            ]),
        ))

if HAS_CE:
    for model_name in tree_like:
        if model_name in models:
            experiments.append((
                model_name,
                "target",
                Pipeline([
                    ("preprocess", make_target_preprocessor(scale_numeric=False)),
                    ("model", models[model_name]),
                ]),
            ))

print("Total experiments:", len(experiments))

display(
    pd.DataFrame(
        [(m, e) for m, e, _ in experiments],
        columns=["model", "encoding"]
    )
)


# =========================
# 8. Run experiments
# =========================

all_rows = []
fitted_pipelines = {}

for model_name, encoding_name, pipe in experiments:
    exp_name = f"{model_name}__{encoding_name}"
    print("Running:", exp_name)

    try:
        rows, fitted_pipe = evaluate_pipeline(model_name, encoding_name, pipe)

        all_rows.extend(rows)
        fitted_pipelines[exp_name] = fitted_pipe

    except Exception as e:
        print("FAILED:", exp_name)
        print(type(e).__name__, e)

results = pd.DataFrame(all_rows)


# =========================
# 9. Reports
# =========================

OUT_DIR = DATA_DIR / "baseline_reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

val_report = (
    results[results["split"] == "val"]
    .sort_values("RMSE_real")
    .reset_index(drop=True)
)

test_original_report = (
    results[results["split"] == "test_original"]
    .sort_values("RMSE_real")
    .reset_index(drop=True)
)

test_without_phantom_report = (
    results[results["split"] == "test_without_phantom_66B"]
    .sort_values("RMSE_real")
    .reset_index(drop=True)
)

scenario_compare_report = (
    results[
        results["split"].isin([
            "test_original",
            "test_without_phantom_66B",
        ])
    ]
    .sort_values(["model", "encoding", "split"])
    .reset_index(drop=True)
)

full_report = (
    results
    .sort_values(["split", "RMSE_real"])
    .reset_index(drop=True)
)

results.to_csv(OUT_DIR / "all_results_with_phantom_scenario.csv", index=False, encoding="utf-8-sig")
val_report.to_csv(OUT_DIR / "val_report.csv", index=False, encoding="utf-8-sig")
test_original_report.to_csv(OUT_DIR / "test_original_report.csv", index=False, encoding="utf-8-sig")
test_without_phantom_report.to_csv(OUT_DIR / "test_without_phantom_report.csv", index=False, encoding="utf-8-sig")
scenario_compare_report.to_csv(OUT_DIR / "phantom_scenario_compare.csv", index=False, encoding="utf-8-sig")
full_report.to_csv(OUT_DIR / "full_report_with_phantom_scenario.csv", index=False, encoding="utf-8-sig")

print("Saved reports to:", OUT_DIR.resolve())

print("Best validation models:")
display(val_report.head(20))

print("Best test original models:")
display(test_original_report.head(20))

print("Best test without Phantom 66B models:")
display(test_without_phantom_report.head(20))


# =========================
# 10. Best model by validation RMSE_real
# =========================

best_row = val_report.iloc[0]
best_name = f"{best_row['model']}__{best_row['encoding']}"

best_pipeline = fitted_pipelines[best_name]

print("Best model by VAL RMSE_real:")
display(best_row.to_frame().T)
print("Pipeline key:", best_name)


# =========================
# 11. Best model scenario comparison
# =========================

best_scenario_compare = (
    results[
        (results["model"] == best_row["model"]) &
        (results["encoding"] == best_row["encoding"]) &
        (results["split"].isin([
            "test_original",
            "test_without_phantom_66B",
        ]))
    ]
    .copy()
    .reset_index(drop=True)
)

print("Best model scenario comparison:")
display(best_scenario_compare)


# =========================
# 12. Show removed Phantom row and prediction
# =========================

phantom_mask = (
    (test_df["brand"] == "rolls royce") &
    (test_df["model"] == "phantom") &
    (test_df["price"] == 66000000000)
)

removed_phantom = test_df[phantom_mask].copy()

print("Removed Phantom rows:", len(removed_phantom))

if len(removed_phantom) > 0:
    removed_pred_log = best_pipeline.predict(
        removed_phantom[feature_cols].copy()
    )

    removed_pred_real = inverse_log_price(removed_pred_log)

    removed_phantom["pred_price_log"] = removed_pred_log
    removed_phantom["pred_price"] = removed_pred_real
    removed_phantom["abs_error_price"] = (
        removed_phantom[REAL_TARGET] - removed_phantom["pred_price"]
    ).abs()
    removed_phantom["ape_price"] = (
        removed_phantom["abs_error_price"] /
        removed_phantom[REAL_TARGET].replace(0, np.nan)
    )

    show_cols = [
        "url",
        "brand",
        "model",
        "name",
        "year",
        "price",
        "price_log",
        "pred_price",
        "pred_price_log",
        "abs_error_price",
        "ape_price",
    ]

    show_cols = [col for col in show_cols if col in removed_phantom.columns]

    display(removed_phantom[show_cols])


# =========================
# 13. Error analysis on original test
# =========================

best_test_pred_log = best_pipeline.predict(X_test)
best_test_pred_real = inverse_log_price(best_test_pred_log)

error_df = test_df.copy()

error_df["pred_price_log"] = best_test_pred_log
error_df["pred_price"] = best_test_pred_real

if REAL_TARGET in error_df.columns:
    error_df["abs_error_price"] = np.abs(
        error_df[REAL_TARGET] - error_df["pred_price"]
    )
    error_df["ape_price"] = (
        error_df["abs_error_price"] /
        error_df[REAL_TARGET].replace(0, np.nan)
    )

error_df["abs_error_log"] = np.abs(
    error_df[TARGET] - error_df["pred_price_log"]
)

ERROR_OUT = OUT_DIR / "test_predictions_best_model_original.csv"
error_df.to_csv(ERROR_OUT, index=False, encoding="utf-8-sig")

print("Saved test predictions to:", ERROR_OUT.resolve())

show_cols = [
    "brand",
    "model",
    "name",
    "year",
    "price",
    "price_log",
    "pred_price",
    "pred_price_log",
    "abs_error_price",
    "ape_price",
    "abs_error_log",
]

show_cols = [col for col in show_cols if col in error_df.columns]

display(
    error_df[show_cols]
    .sort_values(
        "abs_error_price"
        if "abs_error_price" in error_df.columns
        else "abs_error_log",
        ascending=False
    )
    .head(30)
)


# =========================
# 14. Error summary by brand/model
# =========================

if REAL_TARGET in error_df.columns:
    brand_error = (
        error_df
        .groupby("brand")
        .agg(
            count=("brand", "size"),
            mae_real=("abs_error_price", "mean"),
            mape_real=("ape_price", "mean"),
            mae_log=("abs_error_log", "mean"),
        )
        .sort_values("mae_real", ascending=False)
    )

    model_error = (
        error_df
        .groupby(["brand", "model"])
        .agg(
            count=("model", "size"),
            mae_real=("abs_error_price", "mean"),
            mape_real=("ape_price", "mean"),
            mae_log=("abs_error_log", "mean"),
        )
        .query("count >= 10")
        .sort_values("mae_real", ascending=False)
    )

else:
    brand_error = (
        error_df
        .groupby("brand")
        .agg(
            count=("brand", "size"),
            mae_log=("abs_error_log", "mean"),
        )
        .sort_values("mae_log", ascending=False)
    )

    model_error = (
        error_df
        .groupby(["brand", "model"])
        .agg(
            count=("model", "size"),
            mae_log=("abs_error_log", "mean"),
        )
        .query("count >= 10")
        .sort_values("mae_log", ascending=False)
    )

brand_error.to_csv(OUT_DIR / "brand_error_original_test.csv", encoding="utf-8-sig")
model_error.to_csv(OUT_DIR / "model_error_original_test.csv", encoding="utf-8-sig")

display(brand_error.head(30))
display(model_error.head(30))


# =========================
# 15. Save best model
# =========================

import joblib

MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "best_baseline_pipeline.joblib"
joblib.dump(best_pipeline, MODEL_PATH)

print("Saved best model:", MODEL_PATH.resolve())

Train: (17873, 23)
Val:   (2234, 23)
Test:  (2235, 23)
Drop cols: ['url', 'date', 'price']
Target: price_log
Features: 19
Numerical: 8 ['Unnamed: 0', 'volume', 'doors', 'province', 'lat', 'lon', 'odo_win', 'age']
Categorical: 11 ['brand', 'model', 'origin', 'style', 'transmission', 'engine', 'exterior_color', 'interior_color', 'drive', 'name', 'brand_tier']
Detected transform: log1p
Total experiments: 20


,model,encoding
0,linear_regression,onehot_scaled
1,ridge,onehot_scaled
2,lasso,onehot_scaled
3,elasticnet,onehot_scaled
4,knn_5,onehot_scaled
5,knn_15,onehot_scaled
6,decision_tree,ordinal
7,random_forest,ordinal
8,extra_trees,ordinal
9,gradient_boosting,ordinal


Running: linear_regression__onehot_scaled
Running: ridge__onehot_scaled
Running: lasso__onehot_scaled
Running: elasticnet__onehot_scaled
Running: knn_5__onehot_scaled
Running: knn_15__onehot_scaled
Running: decision_tree__ordinal
Running: random_forest__ordinal
Running: extra_trees__ordinal
Running: gradient_boosting__ordinal
Running: hist_gradient_boosting__ordinal
Running: xgboost__ordinal
Running: lightgbm__ordinal
Running: decision_tree__target
Running: random_forest__target
Running: extra_trees__target
Running: gradient_boosting__target
Running: hist_gradient_boosting__target
Running: xgboost__target
Running: lightgbm__target
Saved reports to: D:\car-price-prediction\data\bonbanh\split\baseline_reports
Best validation models:


,model,encoding,split,n_samples,MAE_log,RMSE_log,R2_log,MAE_real,RMSE_real,R2_real
0,extra_trees,target,val,2234,0.063479,0.149281,0.972363,5.598963e+07,1.753824e+08,0.984546
1,extra_trees,ordinal,val,2234,0.062102,0.136241,0.976980,5.779614e+07,1.811643e+08,0.983510
2,xgboost,target,val,2234,0.075672,0.160397,0.968093,7.262639e+07,1.885238e+08,0.982143
3,lightgbm,target,val,2234,0.072857,0.156013,0.969814,7.242180e+07,1.954335e+08,0.980810
4,random_forest,target,val,2234,0.071468,0.169982,0.964166,6.772269e+07,2.089161e+08,0.978071
5,lightgbm,ordinal,val,2234,0.075891,0.134032,0.977720,8.244668e+07,2.156943e+08,0.976625
6,xgboost,ordinal,val,2234,0.080147,0.131336,0.978608,9.074062e+07,2.241428e+08,0.974758
7,ridge,onehot_scaled,val,2234,0.080845,0.155483,0.970018,8.630587e+07,2.335434e+08,0.972597
8,hist_gradient_boosting,target,val,2234,0.086287,0.171082,0.963701,9.056945e+07,2.336871e+08,0.972563
9,linear_regression,onehot_scaled,val,2234,0.079653,0.157104,0.969390,8.703989e+07,2.674292e+08,0.964068


Best test original models:


...

Saved test predictions to: D:\car-price-prediction\data\bonbanh\split\baseline_reports\test_predictions_best_model_original.csv


,brand,model,name,price,price_log,pred_price,pred_price_log,abs_error_price,ape_price,abs_error_log
81,rolls royce,phantom,rolls royce phantom ewb 6.7 v12,66000000000,24.912921,3.577119e+09,21.997823,6.242288e+10,0.945801,2.915097
1148,bentley,mulsanne,bentley mulsanne 6.75 v8,5500000000,22.428014,1.363781e+09,21.033527,4.136219e+09,0.752040,1.394487
826,bentley,bentayga,bentley bentayga 4.0 v8,10888000000,23.110927,7.720000e+09,22.767080,3.168000e+09,0.290963,0.343847
1692,landrover,range rover,landrover range rover sv autobiography lwb 3.0,4200000000,22.158350,1.559626e+09,21.167712,2.640374e+09,0.628660,0.990638
32,porsche,718,porsche 718 boxster style edition 2.0 at,4550000000,22.238393,1.998746e+09,21.415786,2.551254e+09,0.560715,0.822607
1662,toyota,land cruiser,toyota land cruiser exr 3.5 v6,4799000000,22.291673,2.330514e+09,21.569355,2.468486e+09,0.514375,0.722319
1075,landrover,range rover,landrover range rover autobiography lwb 3.0 v6,4999000000,22.332504,2.550959e+09,21.659735,2.448041e+09,0.489706,0.672768
1533,porsche,911,porsche 911 carrera cabriolet,10999000000,23.121070,8.730119e+09,22.890045,2.268881e+09,0.206281,0.231025
443,lamborghini,urus,lamborghini urus 4.0 v8,12800000000,23.272711,1.494836e+10,23.427868,2.148363e+09,0.167841,0.155157
1146,lamborghini,urus,lamborghini urus performante 4.0 v8,16500000000,23.526626,1.447736e+10,23.395852,2.022638e+09,0.122584,0.130774


,count,mae_real,mape_real,mae_log
brand,,,,
rolls royce,4,1.575556e+10,0.262819,0.754960
lamborghini,2,2.085500e+09,0.145212,0.142965
bentley,4,2.066587e+09,0.335486,0.502222
lincoln,1,1.059657e+09,0.278857,0.245967
maserati,2,7.258371e+08,0.123516,0.139038
jaguar,4,5.524266e+08,0.372714,0.481913
cadillac,4,5.220383e+08,0.133622,0.152407
landrover,33,3.808518e+08,0.173084,0.172484
porsche,48,2.235773e+08,0.067705,0.071125


count      mae_real  mape_real   mae_log
brand         model                                                 
landrover     range rover      25  3.767927e+08   0.179477  0.176441
toyota        land cruiser     19  2.461532e+08   0.087843  0.092497
porsche       panamera         13  2.058967e+08   0.055428  0.053210
mercedes benz gls              16  2.014568e+08   0.045910  0.045867
lexus         lx               24  1.838240e+08   0.041780  0.040808
honda         civic            13  1.539509e+08   0.093406  0.149142
mercedes benz s class          50  1.106871e+08   0.038547  0.039685
porsche       cayenne          12  1.081232e+08   0.091044  0.082932
              macan            17  9.389099e+07   0.033797  0.033929
bmw           x3               10  7.315592e+07   0.067503  0.064069
mercedes benz e class          35  6.878642e+07   0.069972  0.067300
lexus         rx               32  6.025749e+07   0.028982  0.029331
toyota        prado            18  5.319812e+07   0.061310  0.060266
kia           carnival         38  4.843927e+07   0.037174  0.037900
mercedes benz glc              65  4.759686e+07   0.036926  0.036799
vinfast       vf8              32  4.576086e+07   0.064009  0.065493
mercedes benz c class          65  3.816665e+07   0.068698  0.055471
honda         crv              37  3.759047e+07   0.051858  0.054569
vinfast       lux a 2.0        13  3.701491e+07   0.066552  0.065933
hyundai       santafe          69  3.692860e+07   0.052184  0.050822
kia           sedona           14  3.420994e+07   0.053583  0.055147
              sorento          16  3.293121e+07   0.041352  0.042348
bmw           3 series         16  3.214065e+07   0.053156  0.054982
hyundai       elantra          11  3.131122e+07   0.063774  0.065048
ford          transit          11  2.878994e+07   0.070618  0.068371
toyota        fortuner         48  2.874747e+07   0.046877  0.047544
ford          ranger          101  2.823174e+07   0.048621  0.048605
mitsubishi    attrage          11  2.732667e+07   0.111844  0.100353
vinfast       lux sa 2.0       13  2.526742e+07   0.042316  0.042132
toyota        innova           37  2.498204e+07   0.076240  0.080831

Saved best model: D:\car-price-prediction\data\bonbanh\split\models\best_baseline_pipeline.joblib
